# cvprofiles — construct-validity profiles

An open methods package for **construct-validity profiles**: partial identification over a finite menu of measurement functions, disciplined by a researcher-authored nomological network. The engine returns an admissible measurement set `M*` and a construct-identified range `[L,U]` for a target functional `beta`.

Everything in this notebook is generated inline — no repository files needed, only the installed package. It shows SCORE → RESTRICT → IDENTIFY → REPORT, plus the empty-set contrast.


## Part 1 — synthetic walk-through

In [ ]:
from __future__ import annotations
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

import cvprofiles
from cvprofiles.pipeline import run_profile

print('cvprofiles', cvprofiles.__version__)

### Build a synthetic scores matrix

One row per unit; one column per measure (`m_*`), plus an auxiliary (`v_aux`) and an outcome (`y`). The researcher supplies the menu — the engine never invents measures.

In [ ]:
rng = np.random.default_rng(42)
n = 200
v_aux = rng.normal(size=n)
m_good = 0.8 * v_aux + 0.6 * rng.normal(size=n)   # strongly aligned with the aux
m_weak = 0.45 * v_aux + 0.9 * rng.normal(size=n)  # weakly aligned
m_slop = -0.5 * v_aux + 1.0 * rng.normal(size=n)  # wrong sign vs the aux
y = 0.5 * m_good + rng.normal(size=n)

scores = pd.DataFrame({
    'unit_id': [f'u{i:03d}' for i in range(n)],
    'm_good': m_good,
    'm_weak': m_weak,
    'm_slop': m_slop,
    'v_aux': v_aux,
    'y': y,
})

roles = {
    'unit_id': 'unit_id',
    'measures': ['m_good', 'm_weak', 'm_slop'],
    'aux': ['v_aux'],
    'outcome': 'y',
    'diagnostic': [],
}

# Nomological network R: admissible measures must correlate with v_aux >= 0.35
# and correlate positively with magnitude >= 0.10.
network = {
    'schema_version': '1',
    'name': 'tutorial_synthetic',
    'delta': 0.0,
    'restrictions': [
        {'id': 'r_corr_min_aux', 'type': 'corr_min', 'theta': 0.35, 'params': {'variable': 'v_aux'}},
        {'id': 'r_corr_sign_aux', 'type': 'corr_sign', 'theta': 0.10, 'params': {'variable': 'v_aux', 'sign': 1}},
    ],
}

beta = {'schema_version': '1', 'type': 'corr_y', 'outcome': 'y', 'params': {}}

### Run a profile

Write the inputs to disk (the engine reads files), then run the full SCORE → RESTRICT → IDENTIFY → REPORT composition.

In [ ]:
work = Path(tempfile.mkdtemp(prefix='cvp_tutorial_'))
scores.to_csv(work / 'scores.csv', index=False)
(work / 'roles.json').write_text(json.dumps(roles))
(work / 'network.yaml').write_text(yaml.safe_dump(network))
(work / 'beta.yaml').write_text(yaml.safe_dump(beta))

result = run_profile(
    scores=work / 'scores.csv',
    roles=work / 'roles.json',
    network=work / 'network.yaml',
    beta=work / 'beta.yaml',
    out_dir=work / 'run',
    seed=0,
    title='Synthetic walk-through',
)

print('run_id :', result.run_id)
print('M*     :', result.identify.admissible)
print('[L,U]  :', result.identify.range_L, result.identify.range_U)
print('rejected:', result.identify.rejected)
print('report :', result.report.html_path)

The designed-valid measures (`m_good`, `m_weak`) survive; the wrong-sign `m_slop` is rejected and its beta never enters `[L,U]`.

### Empty admissible set is a clean result

If the theory + data reject the whole menu, that is a finding, not a crash — the run exits 0 and the report explains the binding bars.

In [ ]:
network_harsh = {
    **network,
    'name': 'tutorial_harsh',
    'restrictions': [
        {'id': 'r_corr_min_aux', 'type': 'corr_min', 'theta': 0.99, 'params': {'variable': 'v_aux'}},
    ],
}
(work / 'network_harsh.yaml').write_text(yaml.safe_dump(network_harsh))

harsh = run_profile(
    scores=work / 'scores.csv',
    roles=work / 'roles.json',
    network=work / 'network_harsh.yaml',
    beta=work / 'beta.yaml',
    out_dir=work / 'harsh',
    seed=0,
    title='Empty-set contrast',
)

print('empty:', harsh.identify.empty)
print('M*   :', harsh.identify.admissible)
print('[L,U]:', harsh.identify.range_L, harsh.identify.range_U)

## What to look at next

- Each run writes an audit trail: `report.html` (human-readable), `report.json` (machine-complete), slacks, admissible set, range, and — when requested — bootstrap / θ-grid / δ-grid diagnostics and the anchors audit.
- Empty `M*` and wide `[L,U]` are scientific features: they quantify measurement fragility under the stated theory.
- The engine is score-agnostic and model-free. Scoring, the menu, and the nomological network are researcher-owned.
- The flagship empirical example is WVS/GPS patience (`tutorials/cvprofiles_wvs_gps_inputs.ipynb`, `evals/wvs_gps_preferences/`).